# Phase 3 Colab Runner (Block 5 Ablations: Partial Top-2)

This notebook runs a 3-run ablation sweep for Block 5 using `scripts/mert_partial_run.py`.

Locked policy:
- Partial unfreeze only
- `--unfreeze-top-k 2` for all runs
- Keep eval protocol identical to Phase 3

Outputs:
- Per-run snapshots in `artifacts/ablations/`
- Consolidated table at `artifacts/ablation_table.csv`
        


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
        


In [ ]:
# ---- Config (pre-filled) ----
REPO_URL = 'https://github.com/sturner11/mert-music-retrieval.git'
REPO_BRANCH = 'master'
DRIVE_REPO_DIR = '/content/drive/MyDrive/mert-music-retrieval'

GITHUB_TOKEN = ''
HF_TOKEN = ''

try:
    from google.colab import userdata  # type: ignore
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or GITHUB_TOKEN
    HF_TOKEN = userdata.get('HF_TOKEN') or HF_TOKEN
except Exception:
    pass

print('Repo:', REPO_URL)
print('Branch:', REPO_BRANCH)
print('Drive dir:', DRIVE_REPO_DIR)
print('GitHub token loaded:', bool(GITHUB_TOKEN))
print('HF token loaded:', bool(HF_TOKEN))
        


In [ ]:
import os
import shutil
import subprocess

os.makedirs(os.path.dirname(DRIVE_REPO_DIR), exist_ok=True)

def auth_url(url: str, token: str) -> str:
    if token and url.startswith('https://'):
        return url.replace('https://', f'https://{token}@')
    return url

clone_url = auth_url(REPO_URL, GITHUB_TOKEN)

if os.path.exists(DRIVE_REPO_DIR) and not os.path.exists(os.path.join(DRIVE_REPO_DIR, '.git')):
    backup_dir = DRIVE_REPO_DIR + '_backup_before_clone'
    if os.path.exists(backup_dir):
        shutil.rmtree(backup_dir)
    shutil.move(DRIVE_REPO_DIR, backup_dir)
    print('Moved non-git folder to:', backup_dir)

if not os.path.exists(DRIVE_REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, DRIVE_REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'remote', 'set-url', 'origin', clone_url], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

print('Repo ready:', DRIVE_REPO_DIR)
        


In [ ]:
import os

os.environ['MMR_PROJECT_ROOT'] = DRIVE_REPO_DIR
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

%cd {DRIVE_REPO_DIR}

!python --version
!pip -q install --upgrade pip
!pip -q install pandas scipy transformers datasets
        


## Manifest Path Remap + Readiness Check
        


In [ ]:
import os
import pandas as pd

ROOT = DRIVE_REPO_DIR
OLD_PREFIX = '/Users/samuelturner/Documents/mert-music-retrieval'

for split in ['train', 'val', 'test']:
    f = f'{ROOT}/artifacts/splits_5s/{split}.csv'
    df = pd.read_csv(f)

    def remap(p):
        p = str(p)
        if p.startswith(OLD_PREFIX + '/'):
            return ROOT + p[len(OLD_PREFIX):]
        if '/data/' in p:
            return ROOT + p[p.index('/data/'):]
        return p

    df['path'] = df['path'].map(remap)
    df.to_csv(f, index=False)

    exists = df['path'].map(os.path.exists)
    print(split, 'existing:', int(exists.sum()), '/', len(df))

expected = {'train': 5584, 'val': 700, 'test': 699}
ok = True
for split, exp in expected.items():
    f = f'{ROOT}/artifacts/splits_5s/{split}.csv'
    df = pd.read_csv(f)
    missing = int((~df['path'].map(os.path.exists)).sum())
    print(f'{split}: rows={len(df)}, expected={exp}, missing={missing}')
    if len(df) != exp or missing != 0:
        ok = False

if not ok:
    raise RuntimeError('Dataset readiness check failed. Fix upload/paths first.')

print('Dataset readiness check passed.')
        


## Ablation Baseline Config (Locked)

All runs share these settings; only `encoder_lr` changes.
        


In [ ]:
import os
import shutil
import pandas as pd

AB_ROOT = 'artifacts/ablations'
os.makedirs(AB_ROOT, exist_ok=True)

BASE_CFG = {
    'device': 'auto',
    'epochs': 15,
    'eval_every_steps': 250,
    'batch_size_audio': 8,
    'batch_size_head': 256,
    'head_dim': 256,
    'head_lr': 1e-3,
    'temperature': 0.07,
    'grad_clip': 1.0,
    'unfreeze_top_k': 2,
}

ABLATIONS = [
    {
        'run_name': 'partial_top2_lr1e5',
        'encoder_lr': 1e-5,
        'hypothesis': 'Reference top-2 unfreeze should match current partial behavior baseline.',
        'change': 'encoder_lr=1e-5; unfreeze_top_k=2',
        'interpretation': 'Pending run.'
    },
    {
        'run_name': 'partial_top2_lr5e6',
        'encoder_lr': 5e-6,
        'hypothesis': 'Lower encoder LR may reduce drift and improve retrieval stability.',
        'change': 'encoder_lr=5e-6; unfreeze_top_k=2',
        'interpretation': 'Pending run.'
    },
    {
        'run_name': 'partial_top2_lr2e5',
        'encoder_lr': 2e-5,
        'hypothesis': 'Higher encoder LR may adapt better but risks overfitting/instability.',
        'change': 'encoder_lr=2e-5; unfreeze_top_k=2',
        'interpretation': 'Pending run.'
    },
]

print('Ablation runs:', [a['run_name'] for a in ABLATIONS])
        


## Run A1: Reference (`encoder_lr=1e-5`)
        


In [ ]:
!python scripts/mert_partial_run.py   --device auto   --epochs 15   --eval-every-steps 250   --batch-size-audio 8   --batch-size-head 256   --head-dim 256   --head-lr 1e-3   --encoder-lr 1e-5   --unfreeze-top-k 2   --grad-clip 1.0   --temperature 0.07
        


In [ ]:
import shutil
run_name = 'partial_top2_lr1e5'
shutil.copyfile('artifacts/mert_partial_results.csv', f'artifacts/ablations/{run_name}_results.csv')
shutil.copyfile('artifacts/mert_partial_rankings.csv', f'artifacts/ablations/{run_name}_rankings.csv')
shutil.copyfile('artifacts/mert_partial_best.pt', f'artifacts/ablations/{run_name}_best.pt')
print('Saved snapshots for', run_name)
        


## Run A2: Lower Encoder LR (`encoder_lr=5e-6`)
        


In [ ]:
!python scripts/mert_partial_run.py   --device auto   --epochs 15   --eval-every-steps 250   --batch-size-audio 8   --batch-size-head 256   --head-dim 256   --head-lr 1e-3   --encoder-lr 5e-6   --unfreeze-top-k 2   --grad-clip 1.0   --temperature 0.07
        


In [ ]:
import shutil
run_name = 'partial_top2_lr5e6'
shutil.copyfile('artifacts/mert_partial_results.csv', f'artifacts/ablations/{run_name}_results.csv')
shutil.copyfile('artifacts/mert_partial_rankings.csv', f'artifacts/ablations/{run_name}_rankings.csv')
shutil.copyfile('artifacts/mert_partial_best.pt', f'artifacts/ablations/{run_name}_best.pt')
print('Saved snapshots for', run_name)
        


## Run A3: Higher Encoder LR (`encoder_lr=2e-5`)
        


In [ ]:
!python scripts/mert_partial_run.py   --device auto   --epochs 15   --eval-every-steps 250   --batch-size-audio 8   --batch-size-head 256   --head-dim 256   --head-lr 1e-3   --encoder-lr 2e-5   --unfreeze-top-k 2   --grad-clip 1.0   --temperature 0.07
        


In [ ]:
import shutil
run_name = 'partial_top2_lr2e5'
shutil.copyfile('artifacts/mert_partial_results.csv', f'artifacts/ablations/{run_name}_results.csv')
shutil.copyfile('artifacts/mert_partial_rankings.csv', f'artifacts/ablations/{run_name}_rankings.csv')
shutil.copyfile('artifacts/mert_partial_best.pt', f'artifacts/ablations/{run_name}_best.pt')
print('Saved snapshots for', run_name)
        


## Build `artifacts/ablation_table.csv`
        


In [ ]:
import os
import pandas as pd

meta = {
    'partial_top2_lr1e5': {
        'hypothesis': 'Reference top-2 unfreeze should match current partial behavior baseline.',
        'change': 'encoder_lr=1e-5; unfreeze_top_k=2',
        'interpretation': 'Use as reference for LR sensitivity.'
    },
    'partial_top2_lr5e6': {
        'hypothesis': 'Lower encoder LR may reduce drift and improve retrieval stability.',
        'change': 'encoder_lr=5e-6; unfreeze_top_k=2',
        'interpretation': 'Compare against reference for stability vs adaptation tradeoff.'
    },
    'partial_top2_lr2e5': {
        'hypothesis': 'Higher encoder LR may adapt better but risks overfitting/instability.',
        'change': 'encoder_lr=2e-5; unfreeze_top_k=2',
        'interpretation': 'Compare against reference for adaptation vs degradation risk.'
    },
}

rows = []
for run_name, m in meta.items():
    path = f'artifacts/ablations/{run_name}_results.csv'
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing results snapshot: {path}')
    df = pd.read_csv(path)
    test_df = df[df['split'] == 'test'].copy()
    test_df['run_name'] = run_name
    test_df['hypothesis'] = m['hypothesis']
    test_df['change'] = m['change']
    test_df['interpretation'] = m['interpretation']
    test_df = test_df[['run_name', 'hypothesis', 'change', 'split', 'task', 'R@1', 'R@5', 'R@10', 'interpretation']]
    rows.append(test_df)

ablation_df = pd.concat(rows, ignore_index=True)
ablation_df.to_csv('artifacts/ablation_table.csv', index=False)
print('Saved:', 'artifacts/ablation_table.csv')
print(ablation_df)
        


## Final Display + Best-Run Selection Rule

Best run rule:
1. Max `same_track R@10`
2. Tie -> max `same_track R@1`
3. Tie -> ascending `run_name`
        


In [ ]:
import pandas as pd

ab = pd.read_csv('artifacts/ablation_table.csv')
print('Ablation table (sorted):')
print(ab.sort_values(['run_name', 'task']).to_string(index=False))

st = ab[ab['task'] == 'same_track'][['run_name', 'R@1', 'R@10']].copy()
best = st.sort_values(['R@10', 'R@1', 'run_name'], ascending=[False, False, True]).iloc[0]
print('
Best run by rule:')
print(best.to_string())
        
